# Notebook 2 — Build Semantic Search

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karthikapv-kv/ai-upskill-layer-3/blob/main/notebooks/02_semantic_search.ipynb)

**Story so far:** embeddings put related text close together (NB1). Now we search the **Acme Pay knowledge base** with them.

**You'll build:** keyword search → semantic search → compare the two.

⏱️ **In session (10 min):** E5, E6 · 🏠 **Take-home:** hybrid search, E4 · Runtime: CPU

## 0. Setup

In [ ]:
%pip install -q sentence-transformers==6.1.0

In [ ]:
import json, os, re, urllib.request
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)
TOP_K = 3

DATA_BASE_URL = "https://raw.githubusercontent.com/karthikapv-kv/ai-upskill-layer-3/main/data"

def data_file(relative_path):
    """Local path to a workshop data file; downloads it from GitHub if needed."""
    for folder in ["data", "../data"]:
        path = os.path.join(folder, relative_path)
        if os.path.exists(path):
            return path
    path = os.path.join("data", relative_path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    urllib.request.urlretrieve(f"{DATA_BASE_URL}/{relative_path}", path)
    return path

def embed_texts(texts):   # carried over from Notebook 1
    return model.encode(texts, normalize_embeddings=True)

### The Acme Pay knowledge base: 18 short engineering docs

In [ ]:
with open(data_file("kb_documents.json")) as f:
    documents = json.load(f)

pd.set_option("display.max_colwidth", 70)
pd.DataFrame(documents)[["document_id", "category", "title"]]

## 1. Keyword search
Score = how many query words appear in the document (ignoring common words like *the*, *my*, *was*).

In [ ]:
STOP_WORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "my", "our", "your", "i", "we", "you",
    "it", "this", "that", "to", "of", "in", "on", "for", "and", "or", "with", "how", "do", "does",
    "did", "what", "why", "when", "can", "should", "has", "have", "me", "very",
}

def tokenize(text):
    """Lowercase words, keeping identifiers like db.pool.max_size or ERR_LEDGER_409 intact."""
    tokens = []
    for word in re.findall(r"[a-z0-9_.\-]+", text.lower()):
        word = word.strip(".-_")
        if word and word not in STOP_WORDS:
            tokens.append(word)
    return tokens

def keyword_search(query, documents, top_k=TOP_K):
    query_terms = set(tokenize(query))
    results = []
    for doc in documents:
        score = len(query_terms & set(tokenize(doc["title"] + " " + doc["content"])))
        if score > 0:
            results.append((score, doc))
    results.sort(key=lambda pair: pair[0], reverse=True)
    return results[:top_k]

def show(results):
    if not results:
        print("   (no results)")
    for rank, (score, doc) in enumerate(results, 1):
        print(f"   {rank}. {doc['document_id']}  {score:>5}  {doc['title']}")

A developer on call types:

In [ ]:
query = "My payment was charged twice."
show(keyword_search(query, documents))

**Expected:** KB-002 (gateway timeouts), KB-003, KB-017, each matching one word. The fix, **KB-001** *Procedure for reversing duplicate debit transactions*, is missing. Why?

In [ ]:
kb001 = documents[0]
print("Query words: ", set(tokenize(query)))
print("Shared with KB-001:", set(tokenize(query)) & set(tokenize(kb001["title"] + " " + kb001["content"])))

**No shared words**, so KB-001 scores 0. Same meaning, different vocabulary.

## 2. Semantic search
Embed every document once, embed the query, and rank by cosine similarity (a dot product, since vectors have length 1).

In [ ]:
doc_embeddings = embed_texts([d["title"] + ". " + d["content"] for d in documents])
print("doc_embeddings:", doc_embeddings.shape)   # (18 documents, 384 numbers each)

def semantic_search(query, documents, doc_embeddings, top_k=TOP_K):
    query_embedding = embed_texts([query])[0]
    similarities = doc_embeddings @ query_embedding
    ranked = np.argsort(-similarities)[:top_k]
    return [(round(float(similarities[i]), 3), documents[i]) for i in ranked]

show(semantic_search(query, documents, doc_embeddings))

**Expected (approx.):** KB-017 **0.487** (incident: customers charged twice) · KB-003 **0.476** (idempotency keys) · **KB-001 0.395** ✅

All three are about duplicate charges, and the fix is found without sharing a word.

### This is vector search
`semantic_search` is **exact vector search** (k-nearest neighbours): score the query against *every* stored vector, sort, keep the top-k. The full ranking:

In [ ]:
scores = doc_embeddings @ embed_texts([query])[0]     # 18 × 384 · 384 → 18 scores
for rank, i in enumerate(np.argsort(-scores), 1):
    print(f"{rank:>2}. {documents[i]['document_id']}  {scores[i]:.3f}  {documents[i]['title']}")

**Expected:** 18 rows, best first; the top 3 match above. Exact search checks every vector: instant for 18, slow for millions. Vector databases (NB4) use approximate indexes to stay fast.

## 3. Side by side: the four developer questions

In [ ]:
def compare(query):
    print(f"🔎 {query}")
    print("  Keyword:")
    show(keyword_search(query, documents))
    print("  Semantic:")
    show(semantic_search(query, documents, doc_embeddings))
    print()

for q in ["My customer was charged twice. What should I investigate?",
          "Our Kafka consumer stopped processing messages.",
          "Why has our API suddenly become very slow?",
          "How do I roll back a failed deployment?"]:
    compare(q)

**Expected:** keyword's #1 is often right, but its #2–#3 are noise matched on one word (e.g. declined cards and refunds for *roll back*). Semantic's runners-up stay on topic.

## 4. Where keyword search wins
Exact identifiers (config keys, topic names, error codes) carry little "meaning" for the model.

In [ ]:
compare("max.poll.interval.ms")
compare("payment-events.DLT")

**Expected:** keyword → **KB-007** and **KB-008** ✅; semantic's top result is wrong for both (KB-010, KB-002).

**Takeaway:** neither wins everywhere, so real systems often **combine** them (section 5).

### ⏱️ E5: Semantic beats keyword (~4 min)
Describe an Acme problem **in your own words**, avoiding the documents' vocabulary.

In [ ]:
# ⏱️ E5: describe an Acme problem in your own words, avoiding the documents' wording
e5_query = ""   # TODO

<details><summary>💡 Hint 1</summary>

Think like a stressed developer: *"the site is really slow"*, *"undo the release"*, *"card got refused"*.

</details>

In [ ]:
if not e5_query:
    print("E5 not finished yet: write a query above.")
else:
    compare(e5_query)

### ⏱️ E6: Keyword beats semantic (~4 min)
Pick an **exact identifier** from a document (look at the table in section 0, or the docs' text).

In [ ]:
# ⏱️ E6: search for an exact identifier that appears in a document
e6_query = ""   # TODO

<details><summary>💡 Hint 1</summary>

Try a header, config key or consumer group: `Retry-After`, `max.poll.records`, `ledger-consumers`.

</details>

In [ ]:
if not e6_query:
    print("E6 not finished yet: write a query above.")
else:
    compare(e6_query)

## 5. Hybrid search: the idea 🏠
Why choose? Blend both scores: **70% semantic + 30% keyword** (keyword scaled to 0–1).

In [ ]:
def hybrid_search(query, documents, doc_embeddings, top_k=TOP_K, semantic_weight=0.7):
    query_terms = set(tokenize(query))
    keyword = np.array([len(query_terms & set(tokenize(d["title"] + " " + d["content"]))) for d in documents], dtype=float)
    if keyword.max() > 0:
        keyword = keyword / keyword.max()
    semantic = doc_embeddings @ embed_texts([query])[0]
    combined = semantic_weight * semantic + (1 - semantic_weight) * keyword
    ranked = np.argsort(-combined)[:top_k]
    return [(round(float(combined[i]), 3), documents[i]) for i in ranked]

def rank_of(expected_id, results):
    ids = [doc["document_id"] for _, doc in results]
    return ids.index(expected_id) + 1 if expected_id in ids else "–"

checks = {
    "max.poll.interval.ms": "KB-007",
    "payment-events.DLT": "KB-008",
    "Retry-After": "KB-010",
    "How do I undo a bad release quickly?": "KB-015",
    "The checkout page is really sluggish for everyone": "KB-009",
    "My payment was charged twice.": "KB-001",
}
n = len(documents)
pd.DataFrame([{
    "query": q, "should find": expected,
    "keyword rank": rank_of(expected, keyword_search(q, documents, top_k=n)),
    "semantic rank": rank_of(expected, semantic_search(q, documents, doc_embeddings, top_k=n)),
    "hybrid rank": rank_of(expected, hybrid_search(q, documents, doc_embeddings, top_k=n)),
} for q, expected in checks.items()])

**Expected:** hybrid ranks every identifier 1st (where semantic failed) and keeps *"undo a bad release"* 1st (where keyword failed). But it's not free: docs sharing **no** words with the query drop (KB-009: 1st → 4th, KB-001: 3rd → 4th).

The 70/30 weight is a tuning knob. Real systems use smarter keyword scoring (BM25) and tuned blending, covered in the **vector databases session**.

## 🏠 E4: Add a new document
Acme writes a new runbook. Make it searchable.

In [ ]:
new_document = {
    "document_id": "KB-019",
    "title": "Renewing an expired TLS certificate",
    "category": "api",
    "content": "If the TLS certificate on the edge-gateway expires, HTTPS connections fail: browsers show a "
               "security warning and API clients get certificate errors. Check the expiry date with acmectl "
               "certs list. Renew it with acmectl certs renew edge-gateway and confirm the new date. "
               "Certificates last 90 days and renew automatically 30 days before expiry, so an expired "
               "certificate means the renewal job failed; check its logs.",
}

In [ ]:
# TODO: add new_document to documents, then recompute doc_embeddings

<details><summary>💡 Hint 1</summary>

`documents.append(...)`, then re-run the `embed_texts(...)` line from section 2. New documents must be embedded before they can be found.

</details>

In [ ]:
compare("HTTPS stopped working for all customers")

**Expected after E4:** semantic → **KB-019** first (≈ 0.465); keyword misses it (no shared words).

## Summary
- **Keyword search** matches words: great for identifiers, blind to paraphrases.
- **Semantic search** = embed docs once → embed query → rank by similarity → top-k.
- **Hybrid search** blends both; it helps on identifiers but needs tuning.
- New docs must be **embedded** before they're searchable.

➡️ **Notebook 3:** is MiniLM the right model for Acme? We'll measure.

---
## Solutions
**E5:** e.g. *"The checkout page is really sluggish for everyone"*: keyword → KB-002 ❌, semantic → KB-009 ✅. Or *"How do I undo a bad release quickly?"*: keyword → KB-011 ❌, semantic → KB-015 ✅.

**E6:** e.g. `Retry-After`: keyword → KB-010 ✅, semantic → KB-017 ❌. Or `ledger-consumers`: keyword → KB-006 ✅, semantic → KB-003 ❌.

In [ ]:
#@title E4 solution
if all(d["document_id"] != "KB-019" for d in documents):
    documents.append(new_document)
doc_embeddings = embed_texts([d["title"] + ". " + d["content"] for d in documents])
compare("HTTPS stopped working for all customers")